In [14]:
from pathlib import Path
from langchain_core.documents import Document

documents = []

for file in Path("rag/documents").glob("*.txt"):

    with open(
        file,
        "r",
        encoding="utf-8"
    ) as f:

        text = f.read()

    documents.append(
        Document(
            page_content=text,
            metadata={
                "source": file.name
            }
        )
    )

print("Documents Loaded:", len(documents))

Documents Loaded: 3


In [15]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

In [16]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(
    documents
)

print("Chunks:", len(chunks))

Chunks: 9


In [17]:
for i, chunk in enumerate(chunks[:5]):
    print("=" * 60)
    print("Chunk", i)
    print("Source:", chunk.metadata)
    print(chunk.page_content)

Chunk 0
Source: {'source': 'credit_guidelines.txt'}
Credit Scoring Guidelines

1. Applicants with higher external credit scores are generally considered lower risk borrowers.

2. Applicants with a history of missed payments, delinquencies, or credit defaults are considered higher risk.
Chunk 1
Source: {'source': 'credit_guidelines.txt'}
3. Stable employment history is a positive indicator of repayment capability.

4. Applicants with longer credit histories typically demonstrate more reliable repayment behavior.

5. Credit utilization should be evaluated when assessing financial responsibility.
Chunk 2
Source: {'source': 'credit_guidelines.txt'}
6. A combination of low external credit scores and high debt obligations increases default probability.

7. Younger applicants with limited financial history may require additional verification.

8. Credit risk assessments should combine financial, behavioral, and demographic indicators.
Chunk 3
Source: {'source': 'lending_policies.txt'}
Lending

In [ ]:
from config import GEMINI_API_KEY

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings
)

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001",
    google_api_key=GEMINI_API_KEY
)

print("Embeddings Loaded")

Embeddings Loaded


In [24]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings Loaded")

C:\Users\Asus\AppData\Local\Temp\ipykernel_2240\1823900820.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Asus\OneDrive\Desktop\Explainable-Credit-Risk-System\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings Loaded


In [25]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="rag/vectordb"
)

print("Vector DB Created")

Vector DB Created


In [26]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever Ready")

Retriever Ready


In [27]:
query = "Applicant has high debt to income ratio"

results = retriever.invoke(query)

for doc in results:
    print("=" * 60)
    print(doc.metadata)
    print(doc.page_content)

{'source': 'lending_policies.txt'}
4. Applicants with unstable income sources should receive enhanced risk assessment.

5. Higher loan amounts require stronger evidence of repayment capability.

6. Borrowers with consistent income and repayment history are preferred.
{'source': 'lending_policies.txt'}
Lending Policies

1. Applicants with a debt-to-income ratio greater than 40 percent should undergo manual review.

2. Applicants requesting loans above approved thresholds require additional verification.

3. Employment duration below one year may increase lending risk.
{'source': 'credit_guidelines.txt'}
6. A combination of low external credit scores and high debt obligations increases default probability.

7. Younger applicants with limited financial history may require additional verification.

8. Credit risk assessments should combine financial, behavioral, and demographic indicators.


In [28]:
from google import genai

client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("Gemini Client Ready")

Gemini Client Ready


In [29]:
query = "Applicant has high debt to income ratio"

results = retriever.invoke(query)

context = "\n\n".join(
    [doc.page_content for doc in results]
)

print(context)

4. Applicants with unstable income sources should receive enhanced risk assessment.

5. Higher loan amounts require stronger evidence of repayment capability.

6. Borrowers with consistent income and repayment history are preferred.

Lending Policies

1. Applicants with a debt-to-income ratio greater than 40 percent should undergo manual review.

2. Applicants requesting loans above approved thresholds require additional verification.

3. Employment duration below one year may increase lending risk.

6. A combination of low external credit scores and high debt obligations increases default probability.

7. Younger applicants with limited financial history may require additional verification.

8. Credit risk assessments should combine financial, behavioral, and demographic indicators.


In [30]:
prompt = f"""
You are a credit risk assistant.

Use the following lending policies and risk rules:

{context}

Question:
{query}

Provide a concise explanation.
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

According to Lending Policy 1, a high debt-to-income ratio indicates that the applicant's application should undergo manual review.


In [ ]:
sample_index = 0

print("Probability:", y_prob_xgb[sample_index])

X_test.iloc[sample_index][
    [
        "CREDIT_INCOME_RATIO",
        "ANNUITY_INCOME_RATIO",
        "AGE_YEARS",
        "EMPLOYMENT_YEARS"
    ]
]

NameError: name 'y_prob_xgb' is not defined

In [32]:
from pathlib import Path

for file in Path("models").glob("*"):
    print(file)

models\.gitkeep
